Sanity check for imports:
* backend/db/db.py
* backend/db/models.py

Ecpected Result:
`Tables created: ['tasks', 'state_snapshots']`

In [ ]:
from sqlalchemy import inspect
from backend.db.db import init_db, get_session, engine
from backend.db.models import Task, StateSnapshot

init_db()
inspector = inspect(engine)

print("Tables created:", inspector.get_table_names())


Tables created: ['state_snapshots', 'tasks']


Inserting and reading back as tasks.
Expected: a dict with id: 1, your title/description, done: False, a created_at timestamp, due_at: None.

In [11]:
with get_session() as s:
    t = Task(title="Research deployment options", description="Compare Vercel vs Railway")
    s.add(t)
    s.commit()
    s.refresh(t)
    print(t.to_dict())

{'id': 1, 'title': 'Research deployment options', 'description': 'Compare Vercel vs Railway', 'done': False, 'created_at': '2026-09-10T21:48:02.884692', 'due_at': None}


Query all tasks, confirm to_dict() is JSON-safe
Expected: valid JSON prints with no errors — this is important, it's proof the API layer can return this straight to the frontend later.

In [12]:
import json

with get_session() as s:
    tasks = s.query(Task).all()
    payload = [t.to_dict() for t in tasks]
    print(json.dumps(payload, indent=2))

[
  {
    "id": 1,
    "title": "Research deployment options",
    "description": "Compare Vercel vs Railway",
    "done": false,
    "created_at": "2026-09-10T21:48:02.884692",
    "due_at": null
  }
]


Insert and read back a StateSnapshot


In [13]:
with get_session() as s:
    snap = StateSnapshot(
        active_app="Chrome",
        active_window_title="Digital Workspace Agent - Docs",
        browser_url="https://example.com",
        browser_tab_title="Example Domain",
    )
    s.add(snap)
    s.commit()
    s.refresh(snap)
    print(snap.to_dict())

{'id': 1, 'active_app': 'Chrome', 'active_window_title': 'Digital Workspace Agent - Docs', 'browser_url': 'https://example.com', 'browser_tab_title': 'Example Domain', 'captured_at': '2026-09-10T21:50:39.522432'}


Mark a task done, confirm updates persist

In [14]:
with get_session() as s:
    t = s.query(Task).first()
    t.done = True
    s.commit()

with get_session() as s:
    t = s.query(Task).first()
    print("done =", t.done) 

done = True
